<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/codeAlongs/WRIT20833_VADER_Sentiment_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WRIT 20833 — Sentiment Analysis with VADER

**When Coding Meets Culture: Developing Data-Driven Opinions**

Make an editable copy of this worksheet by going to **File > Save a copy in Drive**

**New to Colab, or submitting?** See [Working in Colab](https://tcu-dcda.github.io/WRIT20833_2026/guides/working-in-colab.html).

**How this notebook works.** Some code cells are blank on purpose:
- **type along** — a comment says what to write; type it with me, then run it.
- **predict first** — write your guess in the comment *before* you run the cell.
- **your turn** — try it on your own.

If a cell gives you an error, read the message — errors are part of the work here, not something to
hide.

In Week 3 you *counted* words — what a text is about. Now we ask a different question: how does
it **feel**? **Sentiment analysis** scores the emotional tone of a text, from negative to positive.
We'll use **VADER**, a tool built for exactly the kind of informal, punctuation-and-caps-heavy language
people use online. And true to this course, we won't just trust its number — we'll check it against our
own reading.

# Setup

VADER lives in a small library we install once. (The `!pip install` line is a one-time setup command,
not Python — run it and move on.)

In [ ]:
!pip install -q vaderSentiment==3.3.2

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
print("VADER ready.")

# Part 1 — How VADER Scores a Text

Hand VADER a string and it returns four numbers. The one you'll use most is **`compound`**: a single
score from **-1 (very negative)** to **+1 (very positive)**, with **0** around neutral. The other three
(`pos`, `neu`, `neg`) show the mix it found.

In [ ]:
scores = analyzer.polarity_scores("This museum exhibition was absolutely AMAZING!!!")
print(scores)
print("compound:", scores["compound"])

**Predict first.** Before running the next cell, guess: which line is most positive? Most
negative? Which is mixed? Then check yourself.

In [ ]:
# my guess — most positive:    most negative:    mixed:
sample_comments = [
    "worst law ever, complete waste of everyone's time",
    "The idea is okay, nothing special but not terrible either",
    "I love love LOVE this, so good for our kids!",
    "It was good but honestly kind of pointless",
]

for comment in sample_comments:
    compound = analyzer.polarity_scores(comment)["compound"]
    print(f"{compound:+.3f}   {comment}")   # :+.3f shows the + or - sign and 3 decimal places

**Check your predictions.** Did the mixed line come out mixed? *"It was good but honestly kind of
pointless"* scores well above zero, higher even than the lukewarm *"okay, nothing special"* line.
Two reasons, and both are about the word list VADER relies on: **pointless** isn't in its list at all, so it
counts for nothing, and **honestly** *is* in the list, rated as a positive word. A lexicon can only hear
the words it was given.

### Sentiment = quantifying connotation
Notice what just happened: VADER turned *feeling* into a *number*. That's **quantifying
connotation** — the emotional coloring of words, beyond their dictionary meaning. "Determined" and
"stubborn" can describe the same behavior but feel different, and VADER scores that difference. It can
because it leans on a **lexicon** — a fixed list of ~7,500 words each pre-rated for sentiment. Caps
and `!!!` nudge the score too. Emoji count only when a word in the emoji's written-out name is on the
list, and for most emoji none is. The lexicon is just a dictionary, so you can look words up
yourself. Not every pair you'd expect made the list.

To browse the whole list, with the ten human ratings behind each word and what VADER makes of every
emoji, open the [VADER Lexicon Explorer](https://tcu-dcda.github.io/WRIT20833_2026/guides/vader-lexicon.html)
on the course site.

In [ ]:
# VADER's lexicon is a dictionary: word -> rating. None means "not in the list" (it counts for nothing)
for word in ["determined", "stubborn", "cheap", "inexpensive"]:
    print(word, analyzer.lexicon.get(word))

### From words to one number
So where does `compound` come from? Let's rebuild one by hand. VADER looks up each word, adds up
the ratings it finds, and then squeezes that total into the range from -1 to +1.

Step one: score the sentence word by word. Two small tools help here.

**The structure:**
```python
word.strip(".,!?")
analyzer.lexicon.get(word, 0)
```
- `.strip(".,!?")` removes any of those punctuation marks from the start or end of a word, so
  `"beautiful,"` becomes `"beautiful"` and matches the lexicon.
- `.get(word, 0)` looks the word up as before, but the second value, `0`, is what you get back when the
  word isn't in the list. A word VADER doesn't know adds nothing to the total.

In [ ]:
sentence = "The museum was beautiful, the guide was friendly, and the cafe was terrible."

total = 0
for word in sentence.split():
    clean_word = word.strip(".,!?").lower()
    rating = analyzer.lexicon.get(clean_word, 0)
    total = total + rating
    print(f"{clean_word:>10}  {rating:+.1f}")

print("\nsum of ratings:", round(total, 3))

Step two: turn the total into `compound`. A sum of ratings can be any size, so VADER
*normalizes* it with a fixed formula:

$$\text{compound} = \frac{\text{total}}{\sqrt{\text{total}^2 + 15}}$$

The 15 is a constant VADER's authors chose. Because the bottom is always a little bigger than the top,
the result always lands between -1 and +1. `math.sqrt()` takes a square root, and `**2` squares a
number.

In [ ]:
import math

by_hand = total / math.sqrt(total**2 + 15)
print("by hand:", round(by_hand, 4))
print("VADER:  ", analyzer.polarity_scores(sentence)["compound"])

The two match. The squeeze also explains why scores pile up near the ends: each extra
positive word moves `compound` less than the one before.

In [ ]:
# the same formula on bigger and bigger totals: it gets close to 1 but never reaches it
for t in [1, 2, 4, 8, 16]:
    print(t, "->", round(t / math.sqrt(t**2 + 15), 4))

### Your turn
That sentence was a clean case: VADER just added the ratings. Change `sentence` in the word-by-word
cell to the line below, rerun the three cells, and compare your hand total with VADER's.

`"The concert was fun and the music was great, but the lines were awful."`

The words carry the same kind of ratings, but VADER's answer won't match yours. Before adding,
VADER adjusts some ratings. Words after **but** count more and words before it count less. Negations
like **not** flip a rating, and intensifiers like **very** or **absolutely**, ALL CAPS, and `!!!`
push it further from zero. Which of those is at work here?

> **About borrowed code.** Nobody writes a sentiment analyzer from scratch — VADER is a
> well-known tool you `pip install`, the way you'd grab one off Stack Overflow (or from an AI today).
> Borrowing it is *expected*, not cheating. We run it here so you can do the part that matters and that
> no library can do for you: **judge whether its scores are any good** on real cultural text.

# Part 2 — Scoring a Whole Dataset

Here's the cleaned comments table from last week — the constructed Texas Ten Commandments examples
(written for the lesson, not collected), with the `stance` you'd label by hand (support / oppose / neutral). Let's have VADER score the **tone** of every
comment at once.

In [ ]:
import pandas as pd

comments_and_stances = [
    ("The Ten Commandments belong in every classroom, period.", "support"),
    ("This is a clear violation of church and state. Keep it out.", "oppose"),
    ("Morals matter and kids today need them more than ever.", "support"),
    ("Whose religion gets to decide? Not the government's job.", "oppose"),
    ("Honestly I have no strong opinion either way.", "neutral"),
    ("God and country, that's what built this nation.", "support"),
    ("Public schools serve everyone, not just one faith.", "oppose"),
    ("Put the Constitution in classrooms, not commandments.", "oppose"),
    ("Finally some common sense values in our schools.", "support"),
    ("Freedom of religion means freedom from it too.", "oppose"),
    ("My kids should learn this at home, not at school.", "neutral"),
]

comments_df = pd.DataFrame(comments_and_stances, columns=["comment", "stance"])
comments_df

Score one first, to see the move on a single row. To pull one value out of a column by its
position, use `.iloc[]`.

**The structure:**
```python
my_df["column_name"].iloc[0]
```
- `my_df["column_name"]` picks out one column, as in Pandas 01.
- `.iloc[0]` takes the value in position 0, the first row. As with a list, counting starts at 0, so
  `.iloc[1]` is the second row.
- The *i* in `iloc` stands for *integer*: you give it a position number.

In [ ]:
# type along — save the first comment as first_comment with .iloc[0], print it, then print its compound score

Now scale up. `.apply()` runs a function on every value in a column and collects the results,
so you don't have to write the loop yourself. We save the results as a new `sentiment` column.

**The structure:**
```python
my_df["new_column"] = my_df["column_name"].apply(function_name)
```
- `function_name` is a function that takes **one value** and returns one value. Here it's
  `get_sentiment`, which takes a comment and returns its compound score.
- Write the function's name **without parentheses**: `.apply(get_sentiment)`, not
  `.apply(get_sentiment())`. You're handing the function to pandas, and pandas calls it once per row.
- The results line up with the original rows, so they go straight into a new column.

HW3 uses this same function, with the same name.

In [ ]:
def get_sentiment(text):
    return analyzer.polarity_scores(text)["compound"]

comments_df["sentiment"] = comments_df["comment"].apply(get_sentiment)
comments_df

Notice the **0.000**s. Several comments score exactly zero, including *"The Ten Commandments belong in
every classroom, period."*, a firm position if there ever was one. VADER isn't calling them neutral. It
found none of their words in its lexicon, so it has nothing to add up. **Here, zero means "no
evidence," not "no opinion."** (Zero can also happen another way: positive and negative words that
cancel out. You'll meet one in Part 5. So when you see a zero, check which kind it is.)

To check a word yourself, search for it in the
[VADER Lexicon Explorer](https://tcu-dcda.github.io/WRIT20833_2026/guides/vader-lexicon.html). If it isn't
there, it adds nothing to the score. The Emoji tab shows the same thing happening to emoji: most of them
score 0 because no word in their description is on the list.

# Part 3 — Reading the Results

With a score per comment, you can ask quantitative questions of the whole crowd.

To find the single most positive comment takes two steps: find *which row* has the highest score, then
pull that row out.

**The structure:**
```python
row_label = my_df["column_name"].idxmax()
my_df.loc[row_label]
```
- `.idxmax()` doesn't give you the highest value. It gives you the **label of the row** where the
  highest value sits. `.idxmin()` does the same for the lowest.
- `.loc[...]` pulls out the whole row with that label, so you get the comment, its stance, and its
  score together. Pick one value from that row by its column name: `row["comment"]`.
- `.loc` finds a row by its **label**; `.iloc` finds it by its **position**. In this table they match
  (0, 1, 2…), but after you sort or filter a table they don't, so pair `.idxmax()` with `.loc`.

In [ ]:
# type along — print the average sentiment (round to 3); then use .idxmax() / .idxmin() with .loc to save the most positive and most negative rows as most_positive and most_negative, and print each score with its comment

### Your turn
`.sort_values("column_name")` puts a table's rows in order by one column, lowest first. Add
`ascending=False` inside the parentheses for highest first.

In [ ]:
# your turn — sort the DataFrame by its sentiment column with .sort_values() (as in Pandas 01) and look at the order

# Part 4 — Tone Is Not the Same as Stance

Here's the critical move. We have two judgments of each comment: the **stance** *you* labeled (a
*position* — for or against the law) and the **tone** *VADER* scored (an *emotion*). It's tempting to
expect them to line up. Let's turn VADER's score into a label and put them side by side.

In [ ]:
def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

comments_df["vader_tone"] = comments_df["sentiment"].apply(label_sentiment)
comments_df[["comment", "stance", "vader_tone"]]   # double brackets: a list of the columns to show

Look for the **mismatches** — and notice they're not VADER "getting it wrong." Stance and tone
are *different axes*. Someone can **oppose** the law in warm, calm language, or **support** it while
sounding furious. A word like "God" or "freedom" reads positive to the lexicon no matter which side is
saying it. Distant reading (VADER's number) and close reading (your judgment) measure different things —
which is exactly why you need both.

Now compare the two judgments across the whole table: what's the average *tone* in each *stance* camp?

**The structure:**
```python
my_df.groupby("group_column")["number_column"].mean()
```
- `.groupby("group_column")` sorts the rows into groups, one for each different value in that column
  (here, one group per stance).
- `["number_column"]` picks the column to measure in each group.
- `.mean()` averages it within each group, so you get one number per group. Add `.round(3)` at the
  end to keep 3 decimal places.

In [ ]:
# type along — the average TONE within each STANCE camp: group by stance, take the mean of sentiment, round to 3

In [ ]:
import matplotlib.pyplot as plt   # plt is the usual short name for matplotlib's plotting tools

average_tone_by_stance = comments_df.groupby("stance")["sentiment"].mean()
average_tone_by_stance.plot(kind="bar", title="Average tone by stance")
plt.ylabel("mean compound sentiment")
plt.axhline(0, color="gray", linewidth=0.8)   # a gray line at 0, so above and below zero stand out
plt.tight_layout()                            # keep the labels from being cut off
plt.show()

# Part 5 — Where VADER Breaks

VADER reads a lexicon, not a mind. It has no idea about sarcasm, irony, or context. **Predict the tone
of each line below, then run it** — and watch it miss.

In [ ]:
# my guess (positive / negative / neutral, one per line):
tricky_comments = [
    "Oh GREAT, more religion forced on our kids.",   # sarcasm: sounds positive, isn't
    "Sure, because schools have NO other problems.",  # sarcasm
    "Honestly? Not a bad idea at all.",               # double negative = positive
    "This is beautifully, tragically un-American.",   # mixed signals
]

for comment in tricky_comments:
    compound = analyzer.polarity_scores(comment)["compound"]
    print(f"{compound:+.3f}   {comment}")

In [ ]:
# the "un-American" line scored 0.000 -- but not because VADER found nothing. Look at the parts:
print(analyzer.polarity_scores("This is beautifully, tragically un-American."))

That zero is a **cancellation**: `pos` and `neg` are equal, so they sum to nothing. Compare the
"no evidence" zeros back in Part 2, where `neu` was 1.0. Same number, different meaning.

Where did VADER's number disagree with your reading? Those gaps are the most interesting
findings you'll report — not failures of the tool, but the boundary where counting stops and
interpretation has to take over.

# Putting It All Together: Score, Compare, Question

The full workflow in one place — exactly what you'll run on **your own** data in HW3: score every
comment, find the extremes, and check the machine's tone against a human's stance.

One new piece here: a filter with more than one condition.

**The structure:**
```python
my_df[(my_df["column_a"] == "x") & (my_df["column_b"] == "y")]
```
- `&` means **and**: keep a row only if both conditions are true. `|` means **or**: keep it if
  either one is.
- Put **each condition in its own parentheses**. Without them, Python reads the line in the wrong
  order and gives an error.
- The cell below combines both: (oppose **and** positive tone) **or** (support **and** negative tone).

In [ ]:
comments_df["sentiment"] = comments_df["comment"].apply(get_sentiment)       # score
comments_df["vader_tone"] = comments_df["sentiment"].apply(label_sentiment)  # label

print("Overall average tone:", round(comments_df["sentiment"].mean(), 3))
print("\nWhere tone and stance disagree:")
mismatches = comments_df[
    ((comments_df["stance"] == "oppose") & (comments_df["vader_tone"] == "positive")) |
    ((comments_df["stance"] == "support") & (comments_df["vader_tone"] == "negative"))
]
# .to_string(index=False) prints the whole table without its row numbers
print(mismatches[["comment", "stance", "vader_tone"]].to_string(index=False))

In [ ]:
# your turn — paste a comment of your own below and score it; does VADER agree with your read?
my_comment = "Type any comment here"
print(analyzer.polarity_scores(my_comment)["compound"])

# Sneak Preview: Where This Is Going

**In HW3** you'll do all of this on a dataset *you* collected: score it, visualize the spread, and
write up where VADER tracked your reading and where it broke. The mismatches are the point.

**After Thanksgiving break (Days 16–17)** comes the last technique: **topic modeling**. Sentiment asked *what
feeling?*; topic modeling asks *what is this whole pile of text even about?* — letting the computer
surface the recurring themes across hundreds of comments at once. Counting, feeling, themes: three
different ways to hear a crowd.

# Playground

In [ ]:
# use this space to experiment!